# Seaquest HF-Expert / Original Critic (retry)

**Question:** if the *unchanged* original `seaquest_ccrl` contrastive-critic pipeline is trained on trajectories from the validated HF CleanRL expert (instead of the old scripted policy), does the critic learn to **use the action input**?

This notebook is a **thin orchestration layer** around the committed modules. It contains NO rewritten critic / sampler / encoder / loss / training loop. It:
1. clones the repo at an explicit commit;
2. mounts Drive (or accepts an uploaded `raw_hf` archive);
3. verifies dataset + frozen-code hashes;
4. trains the committed critic via `train()` (seed 0 only);
5. saves the checkpoint + history;
6. runs the action-use diagnostic;
7. exports one result ZIP.

Only the **dataset source** differs from the original action-blind run.

In [ ]:
# 1. Clone the repo at an EXPLICIT commit (edit COMMIT to the reviewed HF-expert commit).
REPO = 'https://github.com/tingrui-huang/Goal-Conditioned-Confounded-MsPacman'
COMMIT = '1b15467feaa824e3c29ba5e832e36cf5fc4f141b'   # set to the committed HF-expert revision
import os, subprocess
if not os.path.isdir('repo'):
    subprocess.run(['git','clone',REPO,'repo'], check=True)
subprocess.run(['git','-C','repo','checkout',COMMIT], check=True)
os.chdir('repo')
print('HEAD:', subprocess.check_output(['git','rev-parse','HEAD']).decode().strip())

In [ ]:
# 2. Setup: the critic pipeline needs numpy+torch (preinstalled on Colab). get_game() eagerly
#    imports the OCAtari-backed env, so put the repo's VENDORED OC_Atari on the path for BOTH this
#    process AND the training/diagnostic subprocesses (PYTHONPATH). No pip install of ocatari needed.
import sys, os
oca = os.path.abspath('OC_Atari')
sys.path.insert(0, oca)
os.environ['PYTHONPATH'] = oca + os.pathsep + os.environ.get('PYTHONPATH', '')
import torch, ocatari
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| ocatari from', os.path.dirname(ocatari.__file__))

In [ ]:
# 3. Get the HF dataset (raw_hf) FROM GOOGLE DRIVE (your workflow: upload raw_hf.zip to Drive).
#    Drive's web UI cannot unzip -> let Colab unzip it here. NO manual extraction in Drive needed.
import os, glob, zipfile, shutil
DATA = 'seaquest_ccrl/data/raw_hf'
os.makedirs(DATA, exist_ok=True)
have = lambda: len(glob.glob(os.path.join(DATA, 'traj_*.npz')))
if not have():
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP = ''   # optional: hard-set e.g. '/content/drive/MyDrive/raw_hf.zip'
    cands = [ZIP] if ZIP else glob.glob('/content/drive/MyDrive/**/raw_hf*.zip', recursive=True)
    cands = [c for c in cands if c and os.path.isfile(c)]
    if cands:
        print('unzipping', cands[0])
        with zipfile.ZipFile(cands[0]) as zf:
            names = zf.namelist()
            zf.extractall('seaquest_ccrl/data' if any(n.startswith('raw_hf/') for n in names) else DATA)
    elif glob.glob('/content/drive/MyDrive/raw_hf/traj_*.npz'):     # already-extracted folder
        for f in glob.glob('/content/drive/MyDrive/raw_hf/*'): shutil.copy(f, DATA)
    else:                                                            # last resort: direct upload
        from google.colab import files; up = files.upload()
        z = [k for k in up if k.endswith('.zip')][0]
        with zipfile.ZipFile(z) as zf:
            names = zf.namelist()
            zf.extractall('seaquest_ccrl/data' if any(n.startswith('raw_hf/') for n in names) else DATA)
n = have()
print('trajectories at', DATA, ':', n)
assert n == 40, f'expected 40 traj, found {n} -- check the zip path/contents (must be raw_hf/traj_*.npz)'
print('manifest:', os.path.exists(os.path.join(DATA, 'manifest.json')))

In [ ]:
# 4. Verify FROZEN-code + dataset hashes (the experiment must use the unchanged committed pipeline).
import hashlib, json
FROZEN = ['seaquest_ccrl/models/contrastive_critic.py','seaquest_ccrl/models/sa_encoder.py',
          'seaquest_ccrl/models/g_encoder.py','seaquest_ccrl/training/dataset_sampler.py',
          'seaquest_ccrl/training/train_critic.py','seaquest_ccrl/training/config.py',
          'seaquest_ccrl/data/dataset.py','seaquest_ccrl/data/masking.py']
sha = lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()[:16]
for f in FROZEN: print(sha(f), f)
man = json.load(open('seaquest_ccrl/data/raw_hf/manifest.json'))
print('dataset: n_episodes', man['n_episodes'], 'n_transitions', man['n_transitions'],
      'teacher_ckpt_sha256', man['teacher_ckpt_sha256'][:16])

In [ ]:
# 5. Compatibility gate: load raw_hf through the UNMODIFIED loader + sampler and check shapes/dtypes.
import numpy as np, torch
from seaquest_ccrl.games import get_game
from seaquest_ccrl.training.config import TrainConfig
from seaquest_ccrl.training.dataset_sampler import HindsightSampler
game = get_game('seaquest')
gx0,gx1,gy0,gy1 = game.goal_box
cfg = TrainConfig(steps=50000, seed=0, nb_actions=game.nb_actions, goal_x_lo=gx0, goal_x_hi=gx1,
                  goal_y_lo=gy0, goal_y_hi=gy1, goal_radius=game.eps, frame_stack=game.frame_stack)
s = HindsightSampler(game, oracle=False, cfg=cfg, device='cpu', root='seaquest_ccrl/data/raw_hf')
fr, ac, go = s.sample(cfg.batch_size)
print('batch frames', tuple(fr.shape), fr.dtype, '| actions', tuple(ac.shape), ac.dtype,
      'range', int(ac.min()), int(ac.max()), '| goals', tuple(go.shape), go.dtype)
assert int(ac.max()) < cfg.nb_actions and int(ac.min()) >= 0, 'action id out of range'

In [ ]:
# 6. Train the committed critic (seed 0, masked/naive learner view) via the THIN runner -> train().
#    GPU recommended. Reuses the exact recovered config (steps=50000).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'seaquest_ccrl.scripts.run_hf_expert_critic',
                '--root', 'seaquest_ccrl/data/raw_hf', '--steps', '50000', '--seed', '0',
                '--ckpt-dir', 'seaquest_ccrl/checkpoints/hf_seed0'], check=True)

In [ ]:
# 7. Action-use diagnostic (evaluation only; trains nothing).
subprocess.run([sys.executable, '-m', 'seaquest_ccrl.scripts.eval_hf_action_use',
                '--ckpt', 'seaquest_ccrl/checkpoints/hf_seed0/critic_naive.pt',
                '--root', 'seaquest_ccrl/data/raw_hf',
                '--out', 'artifacts/seaquest/hf_original_critic/action_use_diag.json'], check=True)
print(open('artifacts/seaquest/hf_original_critic/action_use_diag.json').read())

In [ ]:
# 8. Export one result ZIP (checkpoint + history + diagnostic).
import shutil, os
os.makedirs('artifacts/seaquest/hf_original_critic', exist_ok=True)
for f in ['seaquest_ccrl/checkpoints/hf_seed0/critic_naive.pt',
          'seaquest_ccrl/checkpoints/hf_seed0/history_naive.json']:
    if os.path.exists(f): shutil.copy(f, 'artifacts/seaquest/hf_original_critic/')
shutil.make_archive('seaquest_hf_original_critic_result', 'zip', 'artifacts/seaquest/hf_original_critic')
print('wrote seaquest_hf_original_critic_result.zip')
try:
    from google.colab import files; files.download('seaquest_hf_original_critic_result.zip')
except Exception: pass